# Step 3: Data Cleaning and Preprocessing
## Wing Shop Demand Forecasting Project

This notebook handles data cleaning and prepares features for modeling.

**Objectives:**
- Handle missing values
- Detect and handle outliers
- Create lag features for time series
- Split data into training and testing sets

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Setup directories
BASE_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')

print(f"Data Directory: {DATA_DIR}")

Data Directory: d:\CADT\InternshipII\forecasting\data


In [3]:
# Helper function for date features
def get_date_features(df, date_col='date'):
    """Extract date features for modeling"""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['day'] = df[date_col].dt.day
    df['dayofweek'] = df[date_col].dt.dayofweek
    df['weekofyear'] = df[date_col].dt.isocalendar().week.astype(int)
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['is_month_start'] = df[date_col].dt.is_month_start.astype(int)
    df['is_month_end'] = df[date_col].dt.is_month_end.astype(int)
    return df

In [4]:
# Load data
def load_data():
    """Load processed sales data"""
    filepath = os.path.join(DATA_DIR, r'D:\CADT\InternshipII\forecasting\data\processed_sales.csv')
    if os.path.exists(filepath):
        return pd.read_csv(filepath, parse_dates=['date'])
    
    filepath = os.path.join(DATA_DIR, r'D:\CADT\InternshipII\forecasting\data\sample_sales.csv')
    if os.path.exists(filepath):
        return pd.read_csv(filepath, parse_dates=['date'])
    
    return None

df = load_data()

if df is not None:
    print(f"Data loaded: {df.shape}")
else:
    print("Error: No data found. Run previous notebooks first.")

Data loaded: (42775, 16)


## 3.1 Handle Missing Values

In [5]:
print("=" * 60)
print("STEP 3: DATA CLEANING")
print("=" * 60)

print("\n[1] HANDLING MISSING VALUES")
print("-" * 40)

# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

print("Missing values by column:")
for col in df.columns:
    if missing[col] > 0:
        print(f"  {col}: {missing[col]:,} ({missing_pct[col]:.2f}%)")

if missing.sum() == 0:
    print("  No missing values found!")

STEP 3: DATA CLEANING

[1] HANDLING MISSING VALUES
----------------------------------------
Missing values by column:
  transactions: 55 (0.13%)


In [6]:
# Fill missing values
df_clean = df.copy()

# Numeric columns: forward fill then backward fill
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(method='ffill').fillna(method='bfill')

# Categorical columns: fill with mode
cat_cols = df_clean.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Boolean/binary columns: fill with 0
if 'is_holiday' in df_clean.columns:
    df_clean['is_holiday'] = df_clean['is_holiday'].fillna(0).astype(int)

if 'onpromotion' in df_clean.columns:
    df_clean['onpromotion'] = df_clean['onpromotion'].fillna(0)

remaining_missing = df_clean.isnull().sum().sum()
print(f"\nMissing values after cleaning: {remaining_missing}")


Missing values after cleaning: 0


## 3.2 Handle Outliers

In [7]:
print("\n[2] HANDLING OUTLIERS")
print("-" * 40)

column = 'unit_sales'
original_count = len(df_clean)

# IQR method
Q1 = df_clean[column].quantile(0.25)
Q3 = df_clean[column].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (df_clean[column] < lower_bound) | (df_clean[column] > upper_bound)
outlier_count = outliers.sum()

print(f"Q1: {Q1:,.0f}")
print(f"Q3: {Q3:,.0f}")
print(f"IQR: {IQR:,.0f}")
print(f"\nOutliers detected: {outlier_count:,} ({(outlier_count/original_count)*100:.2f}%)")

# Cap outliers instead of removing
df_clean.loc[df_clean[column] < lower_bound, column] = lower_bound
df_clean.loc[df_clean[column] > upper_bound, column] = upper_bound
print(f"Outliers capped to range: [{lower_bound:.0f}, {upper_bound:.0f}]")

# Handle negative sales
neg_count = (df_clean[column] < 0).sum()
if neg_count > 0:
    df_clean.loc[df_clean[column] < 0, column] = 0
    print(f"Negative values set to 0: {neg_count:,}")


[2] HANDLING OUTLIERS
----------------------------------------
Q1: 9
Q3: 272
IQR: 263

Outliers detected: 6,140 (14.35%)
Outliers capped to range: [-386, 667]
Negative values set to 0: 1


## 3.3 Handle Zero Sales

In [8]:
print("\n[3] HANDLING ZERO SALES")
print("-" * 40)

zero_count = (df_clean['unit_sales'] == 0).sum()
print(f"Zero sales records: {zero_count:,} ({(zero_count/len(df_clean))*100:.2f}%)")

# Flag zeros for analysis
df_clean['has_sales'] = (df_clean['unit_sales'] > 0).astype(int)


[3] HANDLING ZERO SALES
----------------------------------------
Zero sales records: 2 (0.00%)


## 3.4 Create Lag Features

In [9]:
print("\n[4] CREATING LAG FEATURES")
print("-" * 40)

df_clean = df_clean.sort_values('date')
lags = [1, 7, 14, 28]

# Determine grouping columns
group_cols = []
if 'store_nbr' in df_clean.columns:
    group_cols.append('store_nbr')

product_col = 'family' if 'family' in df_clean.columns else 'product'
if product_col in df_clean.columns:
    group_cols.append(product_col)

if group_cols:
    # Create lags within groups
    for lag in lags:
        df_clean[f'sales_lag_{lag}'] = df_clean.groupby(group_cols)['unit_sales'].shift(lag)
        print(f"  Created sales_lag_{lag}")
    
    # Rolling averages
    df_clean['sales_rolling_7'] = df_clean.groupby(group_cols)['unit_sales'].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean()
    )
    df_clean['sales_rolling_28'] = df_clean.groupby(group_cols)['unit_sales'].transform(
        lambda x: x.rolling(window=28, min_periods=1).mean()
    )
    print("  Created sales_rolling_7")
    print("  Created sales_rolling_28")
else:
    # No grouping
    for lag in lags:
        df_clean[f'sales_lag_{lag}'] = df_clean['unit_sales'].shift(lag)
        print(f"  Created sales_lag_{lag}")
    
    df_clean['sales_rolling_7'] = df_clean['unit_sales'].rolling(window=7, min_periods=1).mean()
    df_clean['sales_rolling_28'] = df_clean['unit_sales'].rolling(window=28, min_periods=1).mean()
    print("  Created sales_rolling_7")
    print("  Created sales_rolling_28")

# Fill NaN values in lag features
lag_cols = [c for c in df_clean.columns if 'lag' in c or 'rolling' in c]
for col in lag_cols:
    df_clean[col] = df_clean[col].fillna(method='bfill')


[4] CREATING LAG FEATURES
----------------------------------------
  Created sales_lag_1
  Created sales_lag_7
  Created sales_lag_14
  Created sales_lag_28
  Created sales_rolling_7
  Created sales_rolling_28


## 3.5 Train/Test Split

In [10]:
print("\n[5] TRAIN/TEST SPLIT")
print("-" * 40)

test_days = 30
df_sorted = df_clean.sort_values('date')

max_date = df_sorted['date'].max()
split_date = max_date - timedelta(days=test_days)

train = df_sorted[df_sorted['date'] <= split_date]
test = df_sorted[df_sorted['date'] > split_date]

print(f"Training period: {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Testing period: {test['date'].min().date()} to {test['date'].max().date()}")
print(f"Training samples: {len(train):,}")
print(f"Testing samples: {len(test):,}")


[5] TRAIN/TEST SPLIT
----------------------------------------
Training period: 2013-01-02 to 2017-07-16
Testing period: 2017-07-17 to 2017-08-15
Training samples: 41,911
Testing samples: 864


## 3.6 Save Cleaned Data

In [11]:
print("\n[6] SAVING CLEANED DATA")
print("-" * 40)

df_clean.to_csv(os.path.join(DATA_DIR, 'clean_sales.csv'), index=False)
train.to_csv(os.path.join(DATA_DIR, 'train_data.csv'), index=False)
test.to_csv(os.path.join(DATA_DIR, 'test_data.csv'), index=False)

print(f"Saved: clean_sales.csv ({len(df_clean):,} records)")
print(f"Saved: train_data.csv ({len(train):,} records)")
print(f"Saved: test_data.csv ({len(test):,} records)")


[6] SAVING CLEANED DATA
----------------------------------------
Saved: clean_sales.csv (42,775 records)
Saved: train_data.csv (41,911 records)
Saved: test_data.csv (864 records)


In [12]:
# Show final data structure
print("\n" + "=" * 60)
print("DATA CLEANING COMPLETE!")
print("=" * 60)
print(f"\nFinal columns: {list(df_clean.columns)}")

df_clean.head()


DATA CLEANING COMPLETE!

Final columns: ['date', 'store_nbr', 'family', 'unit_sales', 'onpromotion', 'transactions', 'dcoilwtico', 'is_holiday', 'year', 'month', 'day', 'dayofweek', 'weekofyear', 'is_weekend', 'is_month_start', 'is_month_end', 'has_sales', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_7', 'sales_rolling_28']


,date,store_nbr,family,unit_sales,onpromotion,transactions,dcoilwtico,is_holiday,year,month,...,is_weekend,is_month_start,is_month_end,has_sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_7,sales_rolling_28
0,2013-01-02,1,AUTOMOTIVE,2.000,0.0,2111.0,93.14,0,2013,1,...,0,0,0,1,38.029,31.0,194.0,38.029,2.000,2.000
19,2013-01-02,1,SEAFOOD,38.029,0.0,2111.0,93.14,0,2013,1,...,0,0,0,1,38.029,31.0,194.0,38.029,38.029,38.029
18,2013-01-02,1,PREPARED FOODS,47.000,0.0,2111.0,93.14,0,2013,1,...,0,0,0,1,38.029,31.0,194.0,38.029,47.000,47.000
17,2013-01-02,1,POULTRY,247.297,0.0,2111.0,93.14,0,2013,1,...,0,0,0,1,38.029,31.0,194.0,38.029,247.297,247.297
16,2013-01-02,1,PERSONAL CARE,194.000,0.0,2111.0,93.14,0,2013,1,...,0,0,0,1,38.029,31.0,194.0,38.029,194.000,194.000
